# 2.3 — Build Interfaces to Chat with Data

**Exam domain:** Gen AI Functions (Domain 2.0) · **Weight:** 38%

## The problem this solves

A product manager asks the data team the same five questions every Monday, in slightly different words,
and waits a day for each answer. What she wants is to type the question and get the number. What you
want is for her to get it without inventing a number that is not in the warehouse.

Building that is mostly not about the model. It is about deciding what the interface remembers between
turns, what facts it is allowed to use, and who is on the hook for maintaining it.

## What you will be able to do

- Hold a multi-turn conversation, and explain why the model itself remembers nothing
- Keep a conversation inside the context window by pruning, summarising or re-retrieving
- Ground an answer in live query results so the model cannot invent figures
- Choose between raw `AI_COMPLETE`, a Streamlit app, a Cortex Agent and Snowflake CoWork — and say what
  each one costs you in build and maintenance
- Call Cortex Analyst over REST from outside Snowflake with the right role and headers

## Before you start

- Run `setup/dataset.sql`.
- Your role needs `USE AI FUNCTIONS` on the account plus `SNOWFLAKE.CORTEX_USER` (or
  `SNOWFLAKE.AI_FUNCTIONS_USER`), and `USAGE`/`SELECT` on the objects the app reads.
- Notebook 2.1 covers `AI_COMPLETE` and `AI_COUNT_TOKENS`; 2.2 covers Cortex Analyst and Cortex Search,
  which are the tools an agent calls.

📖 **Snowflake documentation for this notebook**
- [AI_COMPLETE (single string)](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
- [COMPLETE (legacy) — the documented conversation array](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)
- [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)
- [Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork)
- [Cortex Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)


---

## The one thing to understand first: the model is stateless

Every call to a language model is independent. Nothing carries over. What feels like memory in a chat
interface is you re-sending the entire conversation on every turn, and the model reading it fresh each
time.

That single fact explains most of the engineering in this notebook:

- The conversation grows every turn, so eventually it does not fit, so you must prune or compress it.
- Every turn re-sends every earlier turn, so the *n*-th message costs more than the first.
- "The bot forgot what I said" is never a model bug. It is a message array you trimmed.

```
User input
    |
[Chat application]   <-- Streamlit in Snowflake / REST client / Snowflake CoWork
    |
Conversation state (message history, pruned or summarised)
    |
AI_COMPLETE  (or a Cortex Agent, which manages tools and state for you)
    |
Response -> append to history -> render
```

### Four ways to build this, in ascending build cost

| Approach | You build | You maintain |
|---|---|---|
| **Snowflake CoWork** | nothing — point it at agents | the agents and their tools |
| **Cortex Agent** via Snowsight, SQL or REST | the agent specification | the spec and its tool grants |
| **Streamlit in Snowflake** | the UI and all conversation state | all of it |
| **Raw `AI_COMPLETE`** | prompt assembly, history, pruning, grounding | all of it |

The trade is control against maintenance. Raw `AI_COMPLETE` lets you decide exactly what the model sees,
and makes you responsible for every failure mode. A Cortex Agent plans, calls tools such as Cortex
Analyst and Cortex Search, reflects on what came back, and answers — you stop owning the orchestration
loop, and you stop being able to change it.

→ [More on Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

### Required privileges for a chat application role

```sql
GRANT USE AI FUNCTIONS ON ACCOUNT                           TO ROLE <app_role>;   -- account privilege
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER                   TO ROLE <app_role>;   -- or AI_FUNCTIONS_USER
GRANT USAGE  ON DATABASE GENAI_STUDY                        TO ROLE <app_role>;
GRANT USAGE  ON SCHEMA   GENAI_STUDY.PUBLIC                 TO ROLE <app_role>;
GRANT SELECT ON TABLE    GENAI_STUDY.PUBLIC.SUPPORT_TICKETS TO ROLE <app_role>;
GRANT USAGE  ON WAREHOUSE COMPUTE_WH                        TO ROLE <app_role>;
-- If the app reaches a Cortex Search service or Cortex Analyst, add:
-- GRANT USAGE ON CORTEX SEARCH SERVICE <svc> TO ROLE <app_role>;
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE <app_role>;
```

→ [More on AI function privileges](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)


In [ ]:
%%sql -r env_setup_1
-- Verify the running role can call AI functions before building the chat app
SHOW GRANTS TO ROLE CURRENT_ROLE();


In [ ]:
%%sql -r env_setup_2
-- Both are required: the account privilege AND one database role
-- GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE MY_APP_ROLE;
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE MY_APP_ROLE;

-- Sanity check
SELECT AI_COMPLETE('llama3.1-8b', 'Reply with exactly one word: OK') AS sanity_check;


---

## Single-turn and multi-turn

| | Single-turn | Multi-turn |
|---|---|---|
| **How** | `AI_COMPLETE(model, 'prompt string')` | a conversation array (legacy `SNOWFLAKE.CORTEX.COMPLETE`), or the history serialised into the prompt |
| **Context** | none; each call is independent | the whole history is re-sent every turn |
| **Use for** | batch enrichment, pipelines | chatbots and assistants |

The `messages` array — objects with `role` and `content`, where `role` is `system`, `user` or
`assistant` — is documented on the **legacy** `SNOWFLAKE.CORTEX.COMPLETE` page, which carries a notice
that the function will be deprecated by the end of 2026. Only one system message is allowed, and it must
come first. The `AI_COMPLETE` reference documents a string prompt or a `PROMPT()` object instead.

So there are two honest ways to do multi-turn today, and the exam expects you to know both:

- **Form A** — the legacy conversation array. It is the documented native message format, and it is on a
  deprecation path.
- **Form B** — serialise the history into a single `AI_COMPLETE` prompt string. Forward-compatible, and
  you are now responsible for formatting the turns yourself.

Either way the array grows:

| Turn | The array |
|---|---|
| 1 | `[system, user]` |
| 2 | `[system, user, assistant, user]` — the model's own reply is appended as `assistant` |
| n | grows every turn, and must be pruned or summarised before it exceeds the context window |

→ [More on the legacy conversation array](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)


In [ ]:
%%sql -r multi_turn_sql_1
-- ============================================================
-- Form A: the legacy SNOWFLAKE.CORTEX.COMPLETE conversation array.
-- roles: 'system' | 'user' | 'assistant'; the system message must come first.
-- The page carries a notice that this legacy function is deprecated by the end of 2026,
-- but it is still the only documented native message-array form.
-- ============================================================
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-8b',
    [{'role': 'system',    'content': 'You are a Snowflake support assistant. Be concise.'},
     {'role': 'user',      'content': 'What are the most common issues in our support tickets?'},
     {'role': 'assistant', 'content': 'Billing discrepancies, app crashes, and shipping delays.'},
     {'role': 'user',      'content': 'Which should we fix first to improve satisfaction?'}],
    {'temperature': 0, 'max_tokens': 512}
) AS turn_2_response;


In [ ]:
%%sql -r multi_turn_sql_2
-- Returns a JSON string: {"choices":[{"messages":"..."}], "created":…, "model":…, "usage":{…}}

-- ============================================================
-- Form B: AI_COMPLETE with the history serialised into the prompt string.
-- ============================================================
SELECT AI_COMPLETE(
    model  => 'llama3.1-8b',
    prompt => 'You are a Snowflake support assistant. Be concise.\n\n'
           || 'User: What are the most common issues in our support tickets?\n'
           || 'Assistant: Billing discrepancies, app crashes, and shipping delays.\n'
           || 'User: Which should we fix first to improve satisfaction?\n'
           || 'Assistant:',
    model_parameters => {'temperature': 0, 'max_tokens': 512}
) AS turn_2_response;

-- Context-window management: every turn re-sends the whole history, so prune or summarise.
-- Budget it first:
-- SELECT AI_COUNT_TOKENS('AI_COMPLETE', 'llama3.1-8b', <assembled_prompt>);


> ### ⚠️ Common misconceptions
>
> **"`AI_COMPLETE` keeps the conversation, so I only send the new message."**
> Nothing is kept. The model sees exactly the prompt you send and nothing else. If a follow-up question
> like "and which of those is oldest?" gets a confused answer, the cause is that "those" was never sent.
> → [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)
>
> **"`AI_COMPLETE` accepts a `[{'role': ..., 'content': ...}]` array like the OpenAI API."**
> The message-array form is documented for the legacy `SNOWFLAKE.CORTEX.COMPLETE`, not for
> `AI_COMPLETE`. Use the legacy function when you want the native array, or build the history into a
> string for `AI_COMPLETE`.
> → [COMPLETE (legacy)](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)
>
> **"`AI_COUNT_TOKENS` will tell me what the turn costs."**
> It estimates **input** tokens only. The output is bounded separately by `max_tokens`, which defaults to
> 4096. A pre-flight check that the prompt fits says nothing about how long the answer will be.
> → [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)
>
> **"A system prompt anywhere in the array is fine as long as it is there."**
> Only one system message is permitted, and it must be first. Appending a fresh system message mid-way
> through a long chat — a common way to try to "remind" the model of its instructions — is not a valid
> array.
> → [COMPLETE (legacy)](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)


In [ ]:
# Multi-turn chat in Python (Snowpark)
# The conversation array is the documented input for the legacy SNOWFLAKE.CORTEX.COMPLETE.
# For AI_COMPLETE, serialise the history into the prompt string instead (see the next pattern below).

from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

SYSTEM_PROMPT = (
    "You are a support data analyst assistant. "
    "Answer questions about support tickets concisely. "
    "If asked for data, refer to GENAI_STUDY.PUBLIC.SUPPORT_TICKETS."
)

def chat(user_message: str, history: list) -> tuple:
    """Send a message and return (assistant_reply, updated_history)."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({"role": "user", "content": user_message})

    # Context-window management: keep the system turn plus the last N pairs
    MAX_TURNS = 10
    if len(messages) > MAX_TURNS * 2 + 1:
        messages = [messages[0]] + messages[-(MAX_TURNS * 2):]

    # Form A: the conversation array, documented for the legacy function
    result = session.sql(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE('llama3.1-8b', PARSE_JSON(?), "
        "{'temperature': 0, 'max_tokens': 512}):choices[0]:messages::STRING AS reply",
        params=[json.dumps(messages)]
    ).collect()[0]["REPLY"]

    history.append({"role": "user",      "content": user_message})
    history.append({"role": "assistant", "content": result})
    return result, history

# Simulate a two-turn conversation
history = []
reply1, history = chat("How many open tickets are critical priority?", history)
print(f"Turn 1: {reply1}")

reply2, history = chat("What categories do they fall into?", history)
print(f"Turn 2: {reply2}")
print(f"\nHistory length: {len(history)} messages")


In [ ]:
# Streamlit in Snowflake — multi-turn chat app scaffold
# Streamlit is preinstalled in Snowflake Notebooks, so you can prototype the widgets here;
# deploy the finished app from Snowsight > Projects > Streamlit so it has its own URL and warehouse.
# Note st.set_page_config's page_title, page_icon and menu_items are not supported in notebooks.

streamlit_code = '''
import streamlit as st
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

st.title("Support Ticket Chat")

# Initialise session state
if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.history  = []

# Render chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# User input
if prompt := st.chat_input("Ask about support tickets..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    # Build the conversation array: system first, then the stored history, then the new question
    msgs = [
        {"role": "system", "content": "You are a support analytics assistant. Be concise."},
    ] + st.session_state.history + [{"role": "user", "content": prompt}]

    reply = session.sql(
        "SELECT SNOWFLAKE.CORTEX.COMPLETE(\'llama3.1-8b\', PARSE_JSON(?)):choices[0]:messages::STRING AS r",
        params=[json.dumps(msgs)]
    ).collect()[0]["R"]

    st.session_state.history.append({"role": "user",      "content": prompt})
    st.session_state.history.append({"role": "assistant", "content": reply})
    st.session_state.messages.append({"role": "assistant", "content": reply})

    with st.chat_message("assistant"):
        st.write(reply)
'''

print("Streamlit app scaffold (save as .py and deploy via Snowsight > Streamlit):")
print(streamlit_code)


---

## Grounding: the difference between a chatbot and a reporting tool

A model asked "how many tickets are open?" will produce a number. It has no way to know whether that
number is right. The fix is not a better prompt — it is to run the query yourself and put the result in
front of the model, with an instruction to use only what it was given.

That is retrieval-augmented generation in miniature: **retrieve, ground, generate**. The next cell does
it with a CTE. Swap the CTE for a Cortex Search call and you are grounding on unstructured text instead;
hand the whole job to Cortex Analyst and it generates and runs the SQL itself.

The trade-off is coverage. A grounded bot that only ever sees open-ticket counts will correctly say "I
do not know" to everything else — which is the right failure, and still a failure the user experiences.
Widening the grounding query widens the prompt, and the tokens come back on every turn.

→ [More on Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)


In [ ]:
%%sql -r grounded_chat
-- Grounded chat: inject live query results into the prompt so the model cannot invent numbers
WITH open_stats AS (
    SELECT
        category,
        COUNT(*)              AS open_count,
        MIN(created_at)::DATE AS oldest_ticket
    FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    WHERE status = 'open'
    GROUP BY category
),
context_string AS (
    SELECT LISTAGG(
        category || ': ' || open_count || ' open tickets (oldest: ' || oldest_ticket || ')',
        ' | '
    ) AS ctx
    FROM open_stats
)
SELECT AI_COMPLETE(
    model  => 'llama3.1-8b',
    prompt => 'You are a support manager. Answer using ONLY the data provided; if the answer is not '
           || 'in the data, say you do not know.\n\nOpen ticket counts: ' || ctx
           || '\n\nQuestion: which category needs immediate attention and why?',
    model_parameters => {'temperature': 0}
) AS recommendation
FROM context_string;

-- This is retrieval-augmented generation in miniature: retrieve -> ground -> generate.
-- Swap the CTE for SNOWFLAKE.CORTEX.SEARCH_PREVIEW(...) to ground on unstructured text instead.


---

## Putting it together

**Scenario.** A product manager wants a chatbot that answers questions about support-ticket trends. It
has to remember context across turns and stay inside the model's token limit.

**Which mechanisms do you use to manage context-window growth?**

### Worked solution

1. **Rolling-window pruning** — keep the system message plus the last N turn pairs.
   ```python
   MAX_TURNS = 10
   if len(messages) > MAX_TURNS * 2 + 1:
       messages = [messages[0]] + messages[-(MAX_TURNS * 2):]
   ```
   Cheap and predictable. It also means the bot genuinely forgets turn 1, which users notice.
2. **Summary compression** — replace the oldest turns with one generated summary, injected as an early
   `assistant` message.
   ```sql
   SELECT AI_COMPLETE('llama3.1-8b',
       'Summarize this conversation in 3 sentences: ' || <prior_turns_as_text>) AS compressed_context;
   ```
   Keeps the gist, costs an extra model call per compression, and quietly loses detail the summary judged
   unimportant.
3. **Pre-flight budgeting** — `AI_COUNT_TOKENS('AI_COMPLETE', '<model>', <assembled_prompt>)` before each
   call. Remember this counts input tokens; `max_tokens` caps output separately.
4. **Ground instead of remember** — re-retrieve the facts each turn rather than carrying them in history.
   A short question plus fresh retrieval is usually cheaper than a long history, and it cannot go stale.
5. **Let the platform do it** — a Cortex Agent manages orchestration and tool state for you.

**One detail worth memorising:** the first argument of `AI_COUNT_TOKENS` is the *function name*, such as
`'AI_COMPLETE'`, not the model.


> ### 🤔 Stop and think
>
> - Pruning history makes the bot forget; summarising it makes the bot remember something slightly wrong.
>   For a support assistant, which failure would you rather explain to a user — and does your answer
>   change for a financial reporting assistant?
> - A Cortex Agent removes the orchestration code you would otherwise own. What can you no longer debug
>   when an answer is wrong, and what would you log to compensate?
> - Grounding on a fixed query makes the bot honest inside a narrow range and useless outside it. Where
>   do you draw that boundary, and how does a user find out they have crossed it?


---

## Talking to Snowflake from outside: the REST interface

When the chat interface is not inside Snowflake — a web app, a Slack bot, an existing product — you call
the REST endpoints instead.

| Purpose | Endpoint |
|---|---|
| Ask a question of structured data | `POST /api/v2/cortex/analyst/message` |
| Embeddings from outside Snowflake | `POST /api/v2/cortex/inference:embed` |
| Agent conversations | the Cortex Agents `agent:run` endpoint |

### Cortex Analyst request body

```json
{
  "messages": [
    { "role": "user",
      "content": [ { "type": "text", "text": "Which product has the highest monthly revenue?" } ] }
  ],
  "semantic_view": "GENAI_STUDY.PUBLIC.PRODUCTS_SEMANTIC",
  "stream": false
}
```

You must supply exactly one of `semantic_view`, `semantic_model_file` (a stage path), `semantic_model`
(inline YAML) or `semantic_models` (an array). With the array, Analyst chooses the most appropriate model
or view **for each query** — it does not combine them. `"stream": true` switches to server-sent events,
which is what gives a chat UI token-by-token output instead of a long pause.

### Headers

```
Authorization: Bearer <token>
X-Snowflake-Authorization-Token-Type: KEYPAIR_JWT     # omit for a standard OAuth token
Content-Type: application/json
```

### Which role

| Need | Database role |
|---|---|
| Analyst over REST | `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_ANALYST_USER` |
| Agents over REST | `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_AGENT_USER` |
| Inference endpoints | `SNOWFLAKE.CORTEX_USER` |

> **Choosing an interface.** In-database batch work → SQL. A UI inside Snowflake → Streamlit calling
> `session.sql(...)`. An external application → REST with key-pair JWT or OAuth. A managed multi-turn
> experience over both structured and unstructured data → a Cortex Agent, or Snowflake CoWork if you
> would rather not build anything at all.

→ [More on the Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)


In [ ]:
%%sql -r rest_role_grants
-- Confirm the role that a REST client will authenticate as can reach Analyst
SHOW GRANTS TO DATABASE ROLE SNOWFLAKE.CORTEX_REST_API_USER;

---

## Snowflake CoWork — the no-build interface

CoWork is a ready-to-use conversational application for business users, built on Cortex Agents, Cortex
Analyst and Cortex Search. You do not build a UI; you build the agents and point it at them. Documents a
user uploads are stored on a personal stage and governed like any other Snowflake data.

This is the end of the build-cost ladder from the top of the notebook. The work does not disappear — it
moves from "write a chat app" to "curate semantic views, search services and agent specifications", which
is the part that determines whether the answers are any good anyway.

→ [More on Snowflake CoWork](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-cowork)

### "Update parameters (the messages array for conversation history)"

The exam guide phrases conversation state this way. The mechanics are the ones from the top of this
notebook: the array grows by one `assistant` message and one `user` message per turn, and you resend all
of it. Roles are `system` (first, if present), `user` and `assistant`, documented on the legacy
`SNOWFLAKE.CORTEX.COMPLETE` page. A Cortex Agent maintains the equivalent state for you, in threads.

> "Memory" in a multi-turn conversation is nothing more than you resending the history on every call.
> That is why the array is the thing you update, and why context-window management — not prompt wording —
> is the real engineering problem in a chatbot.


In [ ]:
%%sql -r conversation_array_growth
-- Growing a conversation array turn by turn, and the pruning rule
-- Turn 3 of a conversation: system + two prior exchanges + the new question
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-8b',
    [{'role': 'system',    'content': 'You are a support analytics assistant. Be concise.'},
     {'role': 'user',      'content': 'How many tickets are open?'},
     {'role': 'assistant', 'content': 'There are 42 open tickets.'},
     {'role': 'user',      'content': 'Which category has the most?'},
     {'role': 'assistant', 'content': 'Technical, with 18.'},
     {'role': 'user',      'content': 'How old is the oldest one?'}],
    {'temperature': 0, 'max_tokens': 256}
) AS turn_3;

-- Pruning rule: keep messages[0] (the system turn) plus the last N pairs.
--   messages = [messages[0]] + messages[-(MAX_TURNS * 2):]
-- Budget it first: AI_COUNT_TOKENS('AI_COMPLETE', '<model>', <assembled_prompt>)

In [ ]:
%%sql -r intelligence_agents
-- What Snowflake CoWork (the feature Snowflake previously called Snowflake Intelligence; the exam study guide may still use the older name) surfaces: the agents your role can use
SHOW AGENTS IN ACCOUNT;

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** Which roles are valid in a conversation array, and what is the rule about the system message?

<details><summary>Show answer</summary>

`system`, `user` and `assistant`. Only one system message may be present, and if it is present it must be
the first element. This is documented on the legacy `SNOWFLAKE.CORTEX.COMPLETE` page — the function that
takes the array — not on the `AI_COMPLETE` reference.

→ [COMPLETE (legacy)](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)

</details>

**2.** What is the first argument of `AI_COUNT_TOKENS`, and does it count input or output tokens?

<details><summary>Show answer</summary>

The first argument is the **function name** as a string, for example `'AI_COMPLETE'`; the model name,
when relevant, is a separate argument. It returns an estimate of **input** tokens. Budgeting a chat turn
therefore tells you whether the prompt fits, not what the reply will add.

→ [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)

</details>

**3.** Which two database roles let a REST client reach Cortex Analyst?

<details><summary>Show answer</summary>

`SNOWFLAKE.CORTEX_USER`, which covers the Cortex features broadly, or `SNOWFLAKE.CORTEX_ANALYST_USER`,
which grants Analyst and nothing else. The narrower role is the better default for a service account
whose only job is answering questions about one semantic view.

→ [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

</details>

**4.** A chatbot answers turn 1 well and turn 5 badly, giving answers about a topic from three turns ago.
What is the most likely cause?

<details><summary>Show answer</summary>

Pruning that kept the wrong messages — most often a rolling window that dropped the recent turns while
holding the system message and some early ones, or an off-by-one in the slice. Because the model is
stateless, its answer reflects exactly the array you sent; it is not "confused", it is answering the
conversation it was given. Log the assembled prompt for a failing turn and the cause is usually obvious.

→ [COMPLETE (legacy)](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)

</details>

**5.** A chat app sends only the newest user message to `AI_COMPLETE` each turn and users complain it has
no memory. What do you change?

<details><summary>Show answer</summary>

Resend the history: either build it into the prompt string for `AI_COMPLETE`, or use the legacy
`SNOWFLAKE.CORTEX.COMPLETE` conversation array. There is no session or thread identifier to attach to an
`AI_COMPLETE` call that would make it remember. If you do not want to own that, a Cortex Agent maintains
conversation threads for you.

→ [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

</details>

**6.** The bot reports "1,200 open tickets" and the real number is 340. The prompt did not include any
query results. What happened, and what fixes it?

<details><summary>Show answer</summary>

The model generated a plausible number because nothing in the prompt contained the real one. Grounding
fixes it: run the aggregate yourself, put the result in the prompt, and instruct the model to answer only
from the data provided and to say so when it cannot. The alternative fix is Cortex Analyst, which
generates and executes SQL rather than recalling figures.

→ [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

</details>

**7.** A REST request to Analyst supplies both `semantic_view` and `semantic_model_file`. Is that valid?

<details><summary>Show answer</summary>

No. Exactly one of `semantic_view`, `semantic_model_file`, `semantic_model` or `semantic_models` must be
supplied. If you want Analyst to have access to several models, the supported way is the `semantic_models`
array — and it picks one per query rather than combining them.

→ [Cortex Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)

</details>

**8.** A conversation is approaching the model's context window. Name two mechanisms that keep it inside,
and one thing each mechanism loses.

<details><summary>Show answer</summary>

Rolling-window pruning keeps the system message and the last N pairs — it loses the early turns
completely, so a constraint the user stated at the start silently stops applying. Summary compression
replaces old turns with a generated summary — it keeps the gist and loses whatever the summariser judged
unimportant, plus it costs an extra model call. A third option avoids the problem rather than managing
it: re-retrieve facts each turn instead of carrying them in history.

→ [AI_COUNT_TOKENS](https://docs.snowflake.com/en/sql-reference/functions/ai_count_tokens)

</details>

**9.** Your interface must be embedded in an existing customer-facing web app. Streamlit in Snowflake, or
REST? What does the choice cost?

<details><summary>Show answer</summary>

REST. Streamlit in Snowflake runs inside Snowsight and is the fast path for an internal tool, but it is
not something you embed in your own product. Going REST means you own authentication — key-pair JWT or
OAuth — network egress, and the UI, in exchange for full control over the experience. The cost people
underestimate is token management: the JWT has to be minted and rotated somewhere.

→ [Cortex Analyst REST API](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/rest-api)

</details>

**10.** A team asks whether to hand-build a chat app or deploy a Cortex Agent. What would move you
towards the agent, and what would move you away?

<details><summary>Show answer</summary>

Towards: the questions span structured and unstructured data, so the orchestration you would otherwise
write is genuinely complex; you want tool use, threads and reflection without owning that loop; the team
is small. Away: you need a specific interaction the agent's loop does not support, you need to inspect or
alter exactly what is sent to the model, or the workload is single-purpose batch enrichment where a chat
loop adds nothing. The honest summary is that you are trading debuggability for delivery speed.

→ [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

</details>

**11.** Every turn re-sends the full history. What does that do to the cost of turn 20 compared with turn
1, and which lever actually helps?

<details><summary>Show answer</summary>

Input tokens grow roughly linearly with the conversation, so turn 20 can cost many times turn 1 even
though the user typed six words. Shortening the *question* barely helps. The levers that do are capping
history length, compressing old turns, and replacing carried context with fresh retrieval. Note that
Cortex Analyst is different: it is billed per message, so a long conversation costs the same per turn
regardless of length.

→ [Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

</details>

**12.** *Connecting to another domain.* An agent-backed chat app must answer both "what does the MSA say
about termination?" and "what was revenue by region last quarter?". Which tools does the agent need, and
what has to exist before it can use them?

<details><summary>Show answer</summary>

A Cortex Search tool for the contracts and a Cortex Analyst tool for the revenue — the pattern built in
2.2. Before the agent works, the search service must exist over chunked, indexed document text with
change tracking enabled on its base objects, and the semantic view must exist with the facts, dimensions
and metrics that define "revenue" and "region". The agent grants are also two-layered: the caller needs
`SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_AGENT_USER`, privileges on the agent object, **and**
privileges on the objects its tools touch.

→ [Cortex Agents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents)

</details>
